# 📘 통계적 가설 검정

**가설 검정**(Hypothesis Testing)은 표본 데이터를 바탕으로
모집단에 대한 주장(가설)이 참인지 거짓인지 판단하는 방법입니다.

**핵심 개념:**
- **귀무가설(H₀)**: 차이가 없다는 가설 (예: 모평균 = 50)
- **대립가설(H₁)**: 차이가 있다는 가설 (예: 모평균 ≠ 50)
- **t값**: 표본평균과 가설값의 차이를 표준오차로 나눈 값
- **p값**: 귀무가설이 참일 때, 이보다 극단적인 결과가 나올 확률

**학습 목표:**
- t검정의 절차 이해 (t값 → p값 → 판정)
- `stats.ttest_1samp()` 사용법
- 시뮬레이션으로 p값의 의미 확인

## 1. 분석 준비 — 데이터 불러오기

정크푸드 무게 데이터를 불러옵니다.
제품 표시 무게는 50g이지만, 실제 무게가 정말 50g일까요?
이것을 검정해 봅시다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  라이브러리 임포트 + 데이터 로드            │
# └─────────────────────────────────────────┘

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 정크푸드 무게 데이터 불러오기
junk_food = pd.read_csv("junk_food_weight.csv")["weight"]
print("정크푸드 무게 데이터:")
print(junk_food.values)
print(f"\n표본 크기: {len(junk_food)}")
print(f"표본평균: {np.mean(junk_food):.4f}")
print(f"표본표준편차: {np.std(junk_food, ddof=1):.4f}")

# 히스토그램으로 분포 확인
plt.figure(figsize=(8, 5))
sns.histplot(junk_food, bins=10, color='steelblue', edgecolor='white')
plt.axvline(50, color='red', linestyle='--', label='표시 무게 50g')
plt.axvline(np.mean(junk_food), color='orange', linestyle='--', label=f'표본평균 {np.mean(junk_food):.2f}g')
plt.title('정크푸드 무게 분포')
plt.xlabel('무게 (g)')
plt.ylabel('빈도')
plt.legend()
plt.tight_layout()
plt.show()

## 2. t값 계산

**t값**은 표본평균이 귀무가설(모평균=50)에서 얼마나 떨어져 있는지를
표준오차 단위로 나타낸 값입니다.

$$t = \frac{\bar{x} - \mu_0}{SE} = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$$

- x̄: 표본평균
- μ₀: 귀무가설의 모평균 (여기서는 50)
- SE: 표준오차

> 💡 t값이 크면 클수록 귀무가설과 멀다는 뜻입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  t값 계산                                │
# │  t = (표본평균 - 귀무가설값) / 표준오차  │
# │  귀무가설: 모평균 = 50g                  │
# └─────────────────────────────────────────┘

# 표본평균
mu = np.mean(junk_food)
print(f"표본평균: {mu:.4f}")

# 자유도
df = len(junk_food) - 1
print(f"자유도: {df}")

# 표준오차
sigma = np.std(junk_food, ddof=1)
se = sigma / np.sqrt(len(junk_food))
print(f"표준오차: {se:.4f}")

# t값
t_value = (mu - 50) / se
print(f"\nt값: {t_value:.4f}")
print(f"\n→ t값이 {t_value:.2f}이라는 것은, 표본평균이 귀무가설(50g)에서")
print(f"  표준오차의 {abs(t_value):.2f}배 떨어져 있다는 의미")

## 3. p값 계산

**p값**은 귀무가설이 참이라고 가정할 때, 관찰된 t값보다 더 극단적인 값이 나올 확률입니다.

- p값이 작으면 → 귀무가설을 기각 (차이가 유의미함)
- p값이 크면 → 귀무가설을 기각하지 않음 (차이가 우연일 수 있음)

> 💡 보통 **유의수준 α = 0.05**에서 판정합니다.
> p < 0.05이면 "통계적으로 유의미하다"고 합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  p값 계산                                │
# │  귀무가설이 참일 때 이보다 극단적인 결과가  │
# │  나올 확률                               │
# └─────────────────────────────────────────┘

# p값 직접 계산 (양측 검정)
alpha = stats.t.cdf(t_value, df=df)
p_value = (1 - alpha) * 2
print(f"t값: {t_value:.4f}")
print(f"p값 (직접 계산): {p_value:.6f}")

# scipy의 ttest_1samp 사용 (간편 방법)
result = stats.ttest_1samp(junk_food, 50)
print(f"\n=== stats.ttest_1samp 결과 ===")
print(f"t값: {result.statistic:.4f}")
print(f"p값: {result.pvalue:.6f}")

# 판정
if result.pvalue < 0.05:
    print(f"\np값 {result.pvalue:.6f} < 0.05 → 귀무가설 기각!")
    print(f"→ 모평균은 50g과 통계적으로 유의미하게 다릅니다")
else:
    print(f"\np값 {result.pvalue:.6f} ≥ 0.05 → 귀무가설 채택")
    print(f"→ 모평균이 50g과 다르다고 말할 수 없습니다")

## 4. 시뮬레이션으로 p값 이해하기

p값이 의미하는 바를 시뮬레이션으로 직관적으로 이해해 봅시다.

**귀무가설이 참**(모평균 = 50g)이라고 가정하고,
동일한 표본 크기로 50,000번 표본을 추출해 t값을 계산합니다.
이때 관찰된 t값보다 극단적인 값이 나올 비율이 p값입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  p값 시뮬레이션                          │
# │  귀무가설(모평균=50)이 참일 때           │
# │  관찰된 t값보다 극단적인 결과가 나올 비율  │
# └─────────────────────────────────────────┘

# 표본 정보
size = len(junk_food)
sigma_sim = np.std(junk_food, ddof=1)

# t값을 저장할 배열
t_value_array = np.zeros(50000)

# 귀무가설이 참(모평균=50)이라고 가정
np.random.seed(1)
norm_dist = stats.norm(loc=50, scale=sigma_sim)

for i in range(50000):
    sample = norm_dist.rvs(size=size)
    sample_mean = np.mean(sample)
    sample_se = np.std(sample, ddof=1) / np.sqrt(size)
    t_value_array[i] = (sample_mean - 50) / sample_se

# 시뮬레이션으로 p값 계산
p_sim = (np.sum(t_value_array > t_value) / 50000) * 2
print(f"시뮬레이션 p값: {p_sim:.6f}")
print(f"이론적 p값:     {result.pvalue:.6f}")
print(f"\n→ 귀무가설이 참일 때, 관찰된 t값보다 극단적인 결과가")
print(f"  약 {p_sim*100:.2f}%의 확률로 나옴")
print(f"→ 이 확률이 매우 작으므로 귀무가설을 기각")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  t값 분포 시각화                         │
# │  시뮬레이션 t값 히스토그램 + 기각역       │
# └─────────────────────────────────────────┘

plt.figure(figsize=(10, 6))
# t값 히스토그램
sns.histplot(t_value_array, bins=50, color='steelblue',
             edgecolor='white', stat='density', alpha=0.6)

# 기각역 표시
t_critical = stats.t.ppf(0.975, df=df)
x = np.arange(-6, 6.1, 0.1)
plt.plot(x, stats.t.pdf(x=x, df=df), color='red', linewidth=2,
         label='t분포 (df=19)')
plt.axvline(t_value, color='orange', linewidth=2, linestyle='--',
            label=f'관찰 t값={t_value:.2f}')
plt.axvline(t_critical, color='green', linewidth=1.5, linestyle=':',
            label=f'기각역 ±{t_critical:.2f}')
plt.axvline(-t_critical, color='green', linewidth=1.5, linestyle=':')

plt.title('t값 시뮬레이션 분포와 기각역')
plt.xlabel('t값')
plt.ylabel('밀도')
plt.legend()
plt.tight_layout()
plt.savefig('hypothesis_test.png', dpi=100)
plt.show()

print(f"기각역 임계값: ±{t_critical:.4f}")
print(f"관찰 t값: {t_value:.4f}")
if abs(t_value) > t_critical:
    print(f"→ |t값| > 임계값 → 귀무가설 기각!")
else:
    print(f"→ |t값| ≤ 임계값 → 귀무가설 채택")

# 📋 가설 검정 요약

| 단계 | 내용 | 설명 |
|------|------|------|
| 1. 귀무가설 설정 | H₀: 모평균 = 50g | 차이가 없다는 가설 |
| 2. t값 계산 | t = (x̄ - 50) / SE | 표본평균과 귀무가설의 차이 |
| 3. p값 계산 | 양측 검정 | 귀무가설 하에서 극단적 결과의 확률 |
| 4. 판정 | p < 0.05? | 유의수준에서 기각 여부 결정 |

> 💡 **가설 검정의 핵심**: p값이 작다는 것은 "우연히 이런 결과가 나올 확률이 매우 낮다"는 뜻입니다.
> 따라서 귀무가설(차이가 없다)을 기각하고, "차이가 있다"고 결론내립니다.

## 🎯 연습 문제

1. 위 데이터에서 유의수준을 0.01로 설정했을 때 검정 결과를 확인하세요.
2. 모평균이 52g이라는 귀무가설을 검정하세요 (양측 검정).
3. `stats.ttest_1samp()`를 사용하여 표시 무게 50g에 대한 단측 검정을 수행하세요.
4. 시뮬레이션 횟수를 10,000회로 줄였을 때 p값이 어떻게 변하는지 확인하세요.
5. 표본 크기가 5, 50, 200일 때 각각 t검정을 수행하고, p값의 변화를 관찰하세요.